# BTP step S1c, attempt 3 on Kaggle: E0 only (gemma-2-2b-it)

SAE validation finished in attempt 2 (notebook16b0352a71 v3); its JSON is committed and given here as input (hash-checked).
The E0 collect step crashed there with CUDA error 719 when run without CUDA_LAUNCH_BLOCKING, the same error as attempt 1, while
SAE validation ran 26 layers x 200 traces cleanly with it (`reports/m0/s1c_run_notes.txt`). This run repeats `scripts/s1_sae.sh`
step 3 with the same arguments under CUDA_LAUNCH_BLOCKING=1 (synchronous launches only; results unchanged). No code changes.
**Inputs:** `btp-code` (v3), `btp-s1c-traces` (traces + SAE validation JSON); GPU T4 x2, Internet on, secret `HF_TOKEN`.

In [ ]:
!nvidia-smi
!python --version
import torch
print(torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, 'Need 2 GPUs: set Accelerator to GPU T4 x2'
!df -h /kaggle/working /root /tmp

In [ ]:
# Find the code under /kaggle/input: Kaggle may keep btp_code.zip or auto-extract it into a folder.
import glob, os, shutil, zipfile
DST = '/kaggle/working/Reasoning-Failure-Diagnosis'
shutil.rmtree(DST, ignore_errors=True)
zips = glob.glob('/kaggle/input/**/btp_code.zip', recursive=True)
reqs = [p for p in glob.glob('/kaggle/input/**/requirements.txt', recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(p), 'scripts', 's1_equivalence.sh'))]
if zips:
    print('zip:', zips[0])
    tmp = '/kaggle/working/_unzip'
    shutil.rmtree(tmp, ignore_errors=True)
    zipfile.ZipFile(zips[0]).extractall(tmp)
    inner = [os.path.dirname(p) for p in glob.glob(tmp + '/**/requirements.txt', recursive=True)
             if os.path.exists(os.path.join(os.path.dirname(p), 'scripts', 's1_equivalence.sh'))]
    assert inner, 'no project root inside the zip'
    shutil.copytree(min(inner, key=len), DST)
    shutil.rmtree(tmp)
elif reqs:
    src = os.path.dirname(min(reqs, key=len))
    print('folder:', src)
    shutil.copytree(src, DST)
else:
    raise FileNotFoundError('btp_code not found under /kaggle/input; add the btp-code dataset as input')
assert 'numpy>=2.0' in open(f'{DST}/requirements.txt').read(), 'stale requirements.txt'
print(sorted(os.listdir(DST)))
assert 'hf_reference' in open(f'{DST}/harness/equivalence.py').read(), 'old btp-code version: update the dataset input'
assert 'n_devices' in open(f'{DST}/analysis/sae_validation.py').read(), 'old btp-code version (no S1c 2-GPU patch): update the dataset input'
assert 'Date committed:** 2026-11-11' in open(f'{DST}/prereg/E0.md').read(), 'E0 pre-registration not committed'

In [ ]:
# requirements.txt needs Python 3.13 (TransformerLens 2.x caps numpy<2 on <=3.12), so build a 3.13 venv with uv.
!pip -q install uv
!uv venv --seed --python 3.13 /kaggle/working/venv
!uv pip install --python /kaggle/working/venv/bin/python -r /kaggle/working/Reasoning-Failure-Diagnosis/requirements.txt
os.environ['PATH'] = '/kaggle/working/venv/bin:' + os.environ['PATH']
os.environ['VIRTUAL_ENV'] = '/kaggle/working/venv'
!which python && python -c "import torch, transformer_lens, numpy; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count(), numpy.__version__)"

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')   # never printed
print('HF_TOKEN set:', bool(os.environ['HF_TOKEN']))

In [ ]:
!cd /kaggle/working/Reasoning-Failure-Diagnosis && python -m pytest -q
assert _exit_code == 0, 'pytest failed'

In [ ]:
# 1. Traces from attempt 1, verified by hash
import hashlib
P = '/kaggle/working/Reasoning-Failure-Diagnosis'
srcs = glob.glob('/kaggle/input/**/m0_traces_scored.jsonl', recursive=True)
assert len(srcs) == 1, srcs
os.makedirs(P + '/data/generated/gemma-2-2b-it', exist_ok=True)
TR = P + '/data/generated/gemma-2-2b-it/m0_traces_scored.jsonl'
shutil.copy(srcs[0], TR)
h = hashlib.sha256(open(TR, 'rb').read()).hexdigest()
print(srcs[0], h, sum(1 for _ in open(TR)), 'lines')
assert h == '2f7e58bf4024aeed225e9abe3b6c987f5fbefee9011643d8e2acb86c4d66e776', 'trace file hash mismatch'

In [ ]:
# SAE validation JSON from attempt 2, verified by hash
SV = glob.glob('/kaggle/input/**/sae_validation_gemma-2-2b-it.json', recursive=True)
assert len(SV) == 1, SV
os.makedirs(P + '/reports/m0', exist_ok=True)
shutil.copy(SV[0], P + '/reports/m0/sae_validation_gemma-2-2b-it.json')
h = hashlib.sha256(open(P + '/reports/m0/sae_validation_gemma-2-2b-it.json', 'rb').read()).hexdigest()
print(SV[0], h)
assert h == 'c598c75b8e29f3b59b610ae6cc90abcbc10625ec578ccfce079b155d9ac806ef', 'SAE validation JSON hash mismatch'
import json
print('passing layers:', json.load(open(P + '/reports/m0/sae_validation_gemma-2-2b-it.json'))['passing_layers'])

In [ ]:
!nvidia-smi

In [ ]:
# E0 (s1_sae.sh step 3, same arguments) with CUDA_LAUNCH_BLOCKING=1
!cd /kaggle/working/Reasoning-Failure-Diagnosis && bash -c 'set -o pipefail; (CUDA_LAUNCH_BLOCKING=1 python -m analysis.sparsity_e0 collect --model gemma-2-2b-it --traces data/generated/gemma-2-2b-it/m0_traces_scored.jsonl --sae-report reports/m0/sae_validation_gemma-2-2b-it.json --max-traces 1000 --n-devices 2 --out data/generated/gemma-2-2b-it/e0_tokens.csv.gz && python -m analysis.sparsity_e0 analyse --tokens data/generated/gemma-2-2b-it/e0_tokens.csv.gz --n-layers 26 --out reports/m0/e0_gemma-2-2b-it.json) 2>&1 | tee /kaggle/working/s1c_e0.log'
assert _exit_code == 0 and os.path.exists(P + '/reports/m0/e0_gemma-2-2b-it.json'), 'E0 failed: see /kaggle/working/s1c_e0.log'

In [ ]:
# Collect outputs
OUT = '/kaggle/working/out'
shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
for f in [P + '/reports/m0/e0_gemma-2-2b-it.json', P + '/reports/m0/sae_validation_gemma-2-2b-it.json', '/kaggle/working/s1c_e0.log', P + '/data/generated/gemma-2-2b-it/e0_tokens.csv.gz']:
    shutil.copy(f, OUT)
for f in sorted(os.listdir(OUT)):
    print(os.path.getsize(f'{OUT}/{f}'), f, hashlib.sha256(open(f'{OUT}/{f}', 'rb').read()).hexdigest())
print(open(OUT + '/e0_gemma-2-2b-it.json').read())
!nvidia-smi